# Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.


In [1]:

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()


model = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider="groq"
)

model.invoke("How are you?") # treated as a human message



AIMessage(content='I’m doing great—thanks for asking! How about you?', additional_kwargs={'reasoning_content': 'User asks "How are you?" It\'s a simple greeting. We can respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 75, 'total_tokens': 115, 'completion_time': 0.040711581, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.00351977, 'prompt_tokens_details': None, 'queue_time': 0.081854664, 'total_time': 0.044231351}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e11c8bbf69', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12309-b672-7282-834c-12beacc71800-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 40, 'total_tokens': 115, 'output_token_details': {'reasoning': 18}})

# Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [11]:
from pydantic import BaseModel, Field


class Movie(BaseModel):
    title:str=Field(description="The title of movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie Rating out of 10")



In [12]:
model_with_structure=model.with_structured_output(Movie)

In [13]:
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x10df4d190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10ded9e50>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of movie', '

In [14]:
model.invoke("Provide details about movie HalfGirlfriend")

AIMessage(content='**Half Girlfriend (2017)**  \n*Hindi-language romantic‑drama film directed by Mohit Suri, adapted from Chetan Bhagat’s 2014 novel of the same name.*\n\n| Category | Details |\n|----------|---------|\n| **Director** | Mohit Suri |\n| **Producers** | Arshad Khan (Arshad Khan Productions), Gauri Khan (Gauri Khan Productions) |\n| **Screenplay** | Mohit Suri |\n| **Based on** | *Half Girlfriend* (novel) by Chetan Bhagat |\n| **Music** | Pritam (songs), Amit Trivedi (background score) |\n| **Cinematography** | Aseem Mishra |\n| **Editing** | A. Sreekar Prasad |\n| **Studio** | Arshad Khan Productions, Gauri Khan Productions |\n| **Runtime** | 139 minutes |\n| **Country** | India |\n| **Language** | Hindi |\n| **Release date** | 28\u202fJune\u202f2017 (India) |\n| **Budget** | ₹25 crore (approx. US$3.5\u202fmillion) |\n| **Box‑office** | ₹63.6 crore worldwide (≈\u202fUS$8.9\u202fmillion) |\n\n---\n\n## Plot Summary\n\nThe story follows **Manav Pandey** (Arjun Kapoor), a br

In [15]:
model_with_structure.invoke("Provide details about movie HalfGirlfriend")

Movie(title='Half Girlfriend', year=2017, director='Mohit Suri', rating=6.0)

In [17]:
# Message output along with parsed structured

from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "We need to fetch movie details via the provided function. Use functions.Movie with director, rating, title, year. Probably rating out of 10. We'll call the function.", 'tool_calls': [{'id': 'fc_f7eb49e2-b6e8-4c0c-8f45-77d4f9f5e138', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 164, 'total_tokens': 239, 'completion_time': 0.077374615, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.00794987, 'prompt_tokens_details': None, 'queue_time': 0.241641739, 'total_time': 0.085324485}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12313-e05c-71d2-a4e5-8454be7021ab-0', tool_calls=[

## Nested Structure

In [18]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in rs")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie HalfGirlfriend")
response

MovieDetails(title='Half Girlfriend', year=2017, cast=[Actor(name='Arjun Kapoor', role='Manav Pandey'), Actor(name='Shraddha Kapoor', role='Riya Chaddha'), Actor(name='Pankaj Tripathi', role='Nikhil'), Actor(name='Sushant Singh Rajput', role='Abhishek')], genres=['Comedy', 'Drama', 'Romance'], budget=None)

## TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [19]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [20]:

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in rs")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie HalfGirlfriend")
response

{'budget': 200000000,
 'cast': [{'name': 'Arjun Kapoor', 'role': 'Madhav Bhandari'},
  {'name': 'Shraddha Kapoor', 'role': 'Richa Chadda'},
  {'name': 'Sushant Singh', 'role': 'Kunal'}],
 'genres': ['Romance', 'Drama', 'Musical'],
 'title': 'Half Girlfriend',
 'year': 2017}

# DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [24]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-4o",
    response_format=ContactInfo,  # Auto-selects ProviderStrategy

)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='cc4e8512-6af9-4ecc-af88-7add2cf90e50'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 125, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_bfeb20929a', 'id': 'chatcmpl-EXF5xJB7UNS2IAbK7Gv9if5QVsMYe', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1232d-a2d0-7d82-856

In [25]:
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [26]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')